In [1]:
#%pip install astroquery
#%pip install tqdm
from tqdm import tqdm
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astroquery.ipac.irsa.irsa_dust import IrsaDust
from astropy.coordinates import FK5, SkyCoord
from astroquery.sdss import SDSS
import astropy.units as u

In [2]:
'''
 Caminho até a pasta de trabalho:
 QSOs
'''
save = 'ztffp_processed_data' #Onde serão salvos os dataframes já processados e com filtros

path = 'dados_recebidos_por_email' #Onde estão os dados recebidos por email

output_directory = 'arquivos_de_saida_bandas_g_r_i' # Onde serão salvos o arquivos de saida (pasta "{ra}_{dec}" contendo g, r e i)

path_plots = 'plots' # Onde serão salvos todos os plots

file_path = 'ra_dec_10_SN_20.txt' # Lista .txt com as coordenadas da sua amostr

In [3]:
def mag_to_flux(ab_mag, wl):
    fnu = (10. ** (-0.4 * (48.6 + ab_mag)))
    flam_g = (2.99792458e+18 * fnu) / (wl ** 2.)
    return flam_g

def dmag_to_df(dmag, f):
    df = f * 0.4 * np.log(10) * dmag
    return df

def get_ebv(ra, dec):
    """
    This function return the E(B-V) Galactic extinction in the line of sight at the position of the source.
    Returns
    ----------------
    ebv : float
        E(B-V) Galactic extinction in the line of sight at ra and dec given.
    """
    coo = SkyCoord(ra=float(ra), dec=float(dec), unit='degree', frame=FK5)
    table = IrsaDust.get_query_table(coo, section='ebv')
    ebv = table['ext SandF mean'][0]
    return ebv

def create_ztf_table(path, source_name, ext_cor=True):
    if os.path.exists(os.path.join(path, source_name + '.txt')):
        # Corrigir extincao galactic ou nao?
        if not ext_cor:
            ext_corr = np.array([3.74922728, 2.64227246, 1.92969698]) * 0.0
        if ext_cor:
            ext_corr = np.array([3.74922728, 2.64227246, 1.92969698]) * get_ebv(ra,dec)
        
        
        # Carregando a tablea gigante. Ela tem que ser salva com o nome ztf_"source_name".txt
        index, field, ccdid, qid, filter, pid, infobitssci, sciinpseeing, scibckgnd, scisigpix, zpmaginpsci, zpmaginpsciunc, zpmaginpscirms, clrcoeff, clrcoeffunc, ncalmatches, exptime, adpctdif1, adpctdif2, diffmaglim, zpdiff, programid, jd, rfid, forcediffimflux, forcediffimfluxunc, forcediffimsnr, forcediffimchisq, forcediffimfluxap, forcediffimfluxuncap, forcediffimsnrap, aperturecorr, dnearestrefsrc, nearestrefmag, nearestrefmagunc, nearestrefchi, nearestrefsharp, refjdstart, refjdend, procstatus \
            = np.genfromtxt(os.path.join(path, source_name + '.txt'), delimiter=" ", comments='#', dtype=str, skip_header=57,
                            unpack=True)

        # why/how I'm doing this is described in the tutorial
        nearestrefmag[nearestrefmag == 'null'] = 0
        nearestrefmagunc[nearestrefmagunc == 'null'] = 0
        forcediffimflux[forcediffimflux == 'null'] = 0
        forcediffimfluxunc[forcediffimfluxunc == 'null'] = 0

        nearestrefmag = np.array(nearestrefmag, dtype=float)
        nearestrefmagunc = np.array(nearestrefmagunc, dtype=float)
        zpdiff = np.array(zpdiff, dtype=float)
        forcediffimfluxunc = np.array(forcediffimfluxunc, dtype=float)
        forcediffimflux = np.array(forcediffimflux, dtype=float)
        mjd = np.array(jd, dtype=float) - 2400000.5 # changing from Julian Date to Modified Julian Date

        nearestrefflux = 10 ** (0.4 * (zpdiff - nearestrefmag))
        nearestrefflux[nearestrefmag == 0] = 0
        nearestreffluxunc = nearestrefmagunc * nearestrefflux / 1.0857 
        nearestreffluxunc[nearestrefmag == 0] = 0

        Flux_tot = forcediffimflux + nearestrefflux  # Adding the refernce flux with the diferene photoemtry flux
        Fluxunc_tot = np.sqrt(abs(forcediffimfluxunc ** 2 - nearestreffluxunc ** 2)) # Uncertanty on the flux
        SNR_tot = Flux_tot / Fluxunc_tot
        display(Flux_tot)
    
    
        mag_obs = np.zeros(np.shape(SNR_tot)) 
        mag_err_obs = np.zeros(np.shape(SNR_tot))


        # Calculating photometry in magnitudes opf the AB system for both g and r filters
        for i in range(len(SNR_tot)):
            if SNR_tot[i] > 5:
                if filter[i] == 'ZTF_g':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(Flux_tot[i]) - ext_corr[0]
                    mag_err_obs[i] = 1.0857 / SNR_tot[i]
                if filter[i] == 'ZTF_r':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(Flux_tot[i]) - ext_corr[1]
                    mag_err_obs[i] = 1.0857 / SNR_tot[i]
                if filter[i] == 'ZTF_i':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(Flux_tot[i]) - ext_corr[2]
                    mag_err_obs[i] = 1.0857 / SNR_tot[i]
            else:
                if filter[i] == 'ZTF_g':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(3 * Fluxunc_tot[i]) - ext_corr[0]
                    mag_err_obs[i] = np.nan
                if filter[i] == 'ZTF_r':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(3 * Fluxunc_tot[i]) - ext_corr[1]
                    mag_err_obs[i] = np.nan
                if filter[i] == 'ZTF_i':
                    mag_obs[i] = zpdiff[i] - 2.5 * np.log10(3 * Fluxunc_tot[i]) - ext_corr[2]
                    mag_err_obs[i] = np.nan
        
        out_directory = os.path.join(output_directory, f'{ra}_{dec}')
        os.makedirs(out_directory, exist_ok=True)  # Crie a pasta se ela não existir
        
        # Criando tabela com photometria na band g
        g_cent_wl =  4722.74
        is_g = filter == 'ZTF_g'
        ztf_g = open(os.path.join(out_directory, f'{source_name}_ztf_g.txt'), 'w')
        #ztf_g.write('# if ab_mag_err = nan, the measurement is an upper limit\n')
        ztf_g.write('mjd' + '\t' + 'ab_mag' + '\t' + 'ab_mag_err' + '\t' + 'flux_dens' + '\t' + 'flux_dens_err' + '\n')
        for yy in range(len(mjd[is_g])):
            ztf_g.write('{:.2f}'.format(mjd[is_g][yy]) + '\t' + '{:.2f}'.format(mag_obs[is_g][yy]) + '\t' + '{:.2f}'.format(
                mag_err_obs[is_g][yy]) + '\t' + '{:.2e}'.format(mag_to_flux(mag_obs[is_g][yy], g_cent_wl)) + '\t' + '{:.2e}'.format(dmag_to_df(mag_err_obs[is_g][yy], mag_to_flux(mag_obs[is_g][yy], g_cent_wl))) + '\n')
        ztf_g.close()

        
        # Criando tabela com photometria na band r
        r_cent_wl =  6339.61
        is_r = filter == 'ZTF_r'
        ztf_r = open(os.path.join(out_directory, f'{source_name}_ztf_r.txt'), 'w')
        #ztf_r.write('# if ab_mag_err = nan, the measurement is an upper limit\n')
        ztf_r.write('mjd' + '\t' + 'ab_mag' + '\t' + 'ab_mag_err' + '\t' + 'flux_dens' + '\t' + 'flux_dens_err' + '\n')
        for yy in range(len(mjd[is_r])):
            ztf_r.write('{:.2f}'.format(mjd[is_r][yy]) + '\t' + '{:.2f}'.format(mag_obs[is_r][yy]) + '\t' + '{:.2f}'.format(
                mag_err_obs[is_r][yy]) + '\t' + '{:.2e}'.format(
                mag_to_flux(mag_obs[is_r][yy], r_cent_wl)) + '\t' + '{:.2e}'.format(
                dmag_to_df(mag_err_obs[is_r][yy], mag_to_flux(mag_obs[is_r][yy], r_cent_wl))) + '\n')
        ztf_r.close()

        # Criando tabela com photometria na band i
        i_cent_wl =  7829.03
        is_i = filter == 'ZTF_i'
        ztf_i = open(os.path.join(out_directory, f'{source_name}_ztf_i.txt'), 'w')
        #ztf_i.write('# if ab_mag_err = nan, the measurement is an upper limit\n')
        ztf_i.write('mjd' + '\t' + 'ab_mag' + '\t' + 'ab_mag_err' + '\t' + 'flux_dens' + '\t' + 'flux_dens_err' + '\n')
        for yy in range(len(mjd[is_i])):
            ztf_i.write('{:.2f}'.format(mjd[is_i][yy]) + '\t' + '{:.2f}'.format(mag_obs[is_i][yy]) + '\t' + '{:.2f}'.format(
                mag_err_obs[is_i][yy]) + '\t' + '{:.2e}'.format(
                mag_to_flux(mag_obs[is_i][yy], i_cent_wl)) + '\t' + '{:.2e}'.format(
                dmag_to_df(mag_err_obs[is_i][yy], mag_to_flux(mag_obs[is_i][yy], i_cent_wl))) + '\n')
        ztf_i.close()
        print('Done!')

In [4]:
def plot_lc(output_directory, file_source_name, source_name, offset=1.0, figsize=(10, 13), min_reliability=2, min_observation_points=1): 

    """
    Plota curvas de luz para diferentes bandas aplicando filtros.

    Parâmetros:
    - output_directory: O diretório onde estão os arquivos das bandas.
    - source_name: O nome da fonte.
    - figsize: Tamanho da figura (default: (10, 13)).
    - min_reliability: Corte minimo para realiability.
    - min_observation_points: Corte minimo para o numero de observações.
    - offset: Desvio inf. e sup. da média para o eixo y (mag. média +- offset).
    """

    band_list = ['ztf_g', 'ztf_r', 'ztf_i']
    color_list= ['r', 'g', 'darkorange']

    rms = float(0)
    min_x, max_x = float(0), float(1)
    mjd_combined = pd.Series(dtype=float)

    plt.close('all')
    
    fig, axes = plt.subplots(len(band_list), 1, figsize=figsize, sharex=True)


    for i,band in enumerate(band_list):
        file_path = os.path.join(output_directory + file_source_name, f'{source_name}_{band}.txt')
        print('caminho para bandas no plot',file_path)

        if not os.path.exists(file_path):
            print(f"Arquivo {source_name}_{band}.txt não encontrado. Pulando para a próxima banda.")
            continue
        try:
            mjd, abmag, abmage, flux, fluxe = np.loadtxt(file_path, skiprows=1, unpack=True)

            # Verifique se pelo menos alguns valores não são NaN
            if not np.isnan(abmag).all() and not np.isnan(abmage).all():     

                # Remove valores NaN e substitui os valores repetidos pelo valor médio correspondente
                df_band = pd.DataFrame({'mjd': mjd, 'abmag': abmag, 'abmage': abmage, 'flux': flux, 'fluxe': fluxe})
                df_band = df_band.dropna()
                df_band['abmag'] = df_band.groupby('mjd')['abmag'].transform('mean')
                df_band['abmage'] = df_band.groupby('mjd')['abmage'].transform('mean')
                df_band['flux'] = df_band.groupby('mjd')['flux'].transform('mean')
                df_band['fluxe'] = df_band.groupby('mjd')['fluxe'].transform('mean')
                
                df_band = df_band.drop_duplicates('mjd')  # Remove linhas com mjd duplicado

                # Filtro por quantidade mínima de observações (linhas)
                if len(df_band['mjd']) < min_observation_points:
                    print(f"Menos de {min_observation_points} observações para a banda {band} da coordenada {source_name}. Pulando para a próxima banda.")
                    continue

                # Calcula a média, RMS
                mean_mag = np.mean(df_band['abmag'])
                #mean_magerr = np.mean(df_band['abmage'])
                rmsd_mag = np.sqrt(np.mean((df_band['abmag'] - mean_mag) ** 2))
                
                # Filtro para remover pontos com diferença maior que 4x o RMSD
                outliers_mask = np.abs(df_band['abmag'] - mean_mag) <= 4 * rmsd_mag
                df_band = df_band[outliers_mask]
                
                f_mean_mag = np.mean(df_band['abmag'])
                f_mean_magerr = np.mean(df_band['abmage'])
                f_rmsd_mag = np.sqrt(np.mean((df_band['abmag'] - mean_mag) ** 2))
                
                # Filtro por "Reliability"
                if f_rmsd_mag / f_mean_magerr <= min_reliability:
                    print(f"Reliability menor que {min_reliability} para a banda {band} da coordenada {source_name}. Pulando para a próxima banda.")
                    continue
                
                out_file = os.path.join(save, source_name)
                os.makedirs(out_file, exist_ok=True)
                df_band.to_csv(out_file + f'/{band}.txt', sep='\t', header=False, index=False)

                min_y = f_mean_mag - offset
                max_y = f_mean_mag + offset

                # Usando a band g para definir os limites no eixo x 
                if band == 'ztf_g':
                    mjd_combined = pd.concat([mjd_combined, pd.Series(df_band['mjd'])])
                    rms = f_rmsd_mag 

                    min_x = mjd_combined.min() - 100
                    max_x = mjd_combined.max() + 100

                ax = axes[i]
                ax.errorbar(df_band['mjd'], df_band['abmag'], yerr=df_band['abmage'], color=color_list[i], label=band, fmt='o',ecolor='darkgrey',markeredgecolor='darkgrey',markersize =7)
                
                ax.set_ylabel('Magnitude', fontsize=15)
                ax.set_ylim(min_y,max_y)
                ax.invert_yaxis()
                ax.grid(True)
                ax.legend(title=f"rms: {f_rmsd_mag:.4f}\nReliability: {f_rmsd_mag/f_mean_magerr:.2f}\n")

            else:
                print(f"Todos os valores são NaN para a banda {band} da coordenada {source_name}. Pulando para a próxima banda.")

        except Exception as e:
            print(f"Erro ao plotar o gráfico para a banda {band} da coordenada {source_name}: {str(e)}")
            continue

    plt.xlabel('MJD', fontsize=15)
    plt.xlim(min_x, max_x)

    if output_directory:
        # Verifica se o diretório de saída existe
        if not os.path.exists(path_plots):
            os.makedirs(path_plots)
        
        plt.tight_layout()
        plt.savefig(os.path.join(path_plots, f'{rms:.2f}-{source_name}_plots.png'))
        #print(f"Figura combinada salva em: {os.path.join(path, f'{source_name}_plots.png')}")
    else:
        plt.tight_layout()

#-------------------------------------------------------------------------------------------------------------------------------------------------------#

def plot_hist(output_directory, file_source_name, source_name, min_reliability=2, min_observation_points=1):
    
    band_list = ['ztf_g', 'ztf_r', 'ztf_i']
    
    for i,band in enumerate(band_list):
        file_path = os.path.join(output_directory + file_source_name, f'{source_name}_{band}.txt')

        if not os.path.exists(file_path):
            print(f"Arquivo {source_name}_{band}.txt não encontrado. Pulando para a próxima banda.")
            continue
        try:
            mjd, abmag, abmage, flux, fluxe = np.loadtxt(file_path, skiprows=1, unpack=True)

                # Verifique se pelo menos alguns valores não são NaN
            if not np.isnan(abmag).all() and not np.isnan(abmage).all():    

                # Remove valores NaN e substitui os valores repetidos pelo valor médio correspondente
                df_band = pd.DataFrame({'mjd': mjd, 'abmag': abmag, 'abmage': abmage, 'flux': flux, 'fluxe': fluxe})
                df_band = df_band.dropna()
                df_band['abmag'] = df_band.groupby('mjd')['abmag'].transform('mean')
                df_band['abmage'] = df_band.groupby('mjd')['abmage'].transform('mean')
                df_band['flux'] = df_band.groupby('mjd')['flux'].transform('mean')
                df_band['fluxe'] = df_band.groupby('mjd')['fluxe'].transform('mean')

                df_band = df_band.drop_duplicates('mjd')  # Remove linhas com mjd duplicado

                # Filtro por quantidade mínima de observações (linhas)
                if len(df_band['mjd']) < min_observation_points:
                    print(f"Menos de {min_observation_points} observações para a banda {band} da coordenada {source_name}. Pulando para a próxima banda.")
                    continue
       
                # Calcula a média, RMS e reliability
                mean_mag = np.mean(df_band['abmag'])
                rmsd_mag = np.sqrt(np.mean((df_band['abmag'] - mean_mag) ** 2))
                
                # Filtro para remover pontos com diferença maior que 4x o RMSD
                outliers_mask = np.abs(df_band['abmag'] - mean_mag) <= 4 * rmsd_mag
                df_band = df_band[outliers_mask]
                
                mean_flux = np.mean(df_band['flux'])
                rmsd_flux= np.sqrt(np.mean((df_band['flux'] - mean_flux) ** 2))

                media = np.mean(df_band['abmag'])
                mean_magerr = np.mean(df_band['abmage'])
                rmsd = np.sqrt(np.mean((df_band['abmag'] - media) ** 2))

                reliability = rmsd/mean_magerr

                # Adiciona o valor de rmsd à lista 
                if band == 'ztf_g' and reliability >= min_reliability:
                    rms_g.append(rmsd)
                    flux_g.append(rmsd_flux)
                elif band == 'ztf_r' and reliability >= min_reliability:
                    rms_r.append(rmsd)
                    flux_r.append(rmsd_flux)
                elif band == 'ztf_i' and reliability >= min_reliability:
                    rms_i.append(rmsd)
                    flux_i.append(rmsd_flux)
            else:
                print(f"Todos os valores são NaN para a banda {band} da coordenada {source_name}. Pulando para a próxima banda.")

        except Exception as e:
            print(f"Erro ao plotar o gráfico para a banda {band} da coordenada {source_name}: {str(e)}")
            continue    



In [5]:
def read_coordinates_from_file(file_path):
    ra_list, dec_list = [], []
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                ra, dec = float(parts[0]), float(parts[1])
                ra_list.append(ra)
                dec_list.append(dec)
    return ra_list, dec_list

def create_source_name(ra, dec):
    return f'ztf_ra{ra}_dec{dec}'

def create_file_source_name(ra, dec):
    return f'{ra}_{dec}'

def buscar_redshift_sdss(ra, dec, lista):
    result = SDSS.query_region(SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg), frame='icrs'), spectro=True)
    
    if result is not None and len(result) > 0:
        redshift = result['z'][0]
        if redshift <= 0.5 and redshift >= 0.4:
            print('objeto: RA =',ra,'DEC =',dec, 'Z=',redshift)
            return lista.append(redshift)
        else:
            return None
    else:
        return None

In [6]:

# Caminho para o arquivo de coordenadas
ra_list, dec_list = read_coordinates_from_file(file_path)

flux_g = []
flux_r = []
flux_i = []

rms_g = []
rms_r = []
rms_i = []

In [7]:

# Processa os arquivos para cada par de coordenadas
for ra, dec in tqdm(zip(ra_list, dec_list), total=len(ra_list), desc="Processing"):

    source_name = create_source_name(ra, dec)
    create_ztf_table(path, source_name, ext_cor=True)


Processing:   0%|          | 0/2106 [00:00<?, ?it/s]

array([ 9.13324037e+02,  5.74583168e+02,  9.23743469e+02,  6.29611009e+02,
        8.68257193e+02,  6.64459463e+02,  8.20807828e+02,  6.85854784e+02,
        8.17047824e+02,  5.33570696e+02,  7.90386508e+02,  7.02098483e+02,
        7.34601926e+02,  9.25856852e+02,  6.67902082e+02,  8.42901499e+02,
        7.30042517e+02,  8.09975735e+02,  8.80950225e+02,  6.49078220e+02,
        9.36017782e+02,  5.04368162e+02,  9.65649966e+02,  6.40537253e+02,
        9.13641153e+02,  5.75486829e+02,  3.98545069e+02,  8.68459500e+02,
        9.37572342e+02,  9.32803285e+02,  9.27263452e+02,  7.38311760e+02,
        7.01054181e+02,  9.57415785e+02,  7.06964614e+02,  9.81742519e+02,
        9.64037586e+02,  9.52470491e+02,  1.02107850e+03,  9.57061571e+02,
        9.59363360e+02,  8.90751163e+02,  9.57895986e+02,  1.00742543e+03,
        7.73388375e+02,  7.78867081e+02,  7.78233592e+02,  1.03485064e+03,
        9.16866943e+02, -9.92946963e+04,  6.61643755e+02,  6.19898035e+02,
        6.90371783e+02,  

Done!


array([350.97141656, 267.71500087, 225.61010471, ..., 702.01954059,
       673.41879668, 667.10637048])

Processing:   0%|          | 6/2106 [00:00<01:06, 31.74it/s]

Done!


... (681 outputs left out)

array([532.12235512, 141.43438779, 298.5608305 , ..., 421.16314306,
       526.19473824, 332.82471969])

Processing:  39%|███▉      | 827/2106 [00:42<01:43, 12.31it/s]/tmp/ipykernel_14457/4255097848.py:34: UserWarning: genfromtxt: Empty input file: "dados_recebidos_por_email/ztf_ra156.174197_dec4.344393.txt"
  = np.genfromtxt(os.path.join(path, source_name + '.txt'), delimiter=" ", comments='#', dtype=str, skip_header=57,
Processing:  40%|███▉      | 835/2106 [00:42<01:04, 19.80it/s]

Done!


ValueError: not enough values to unpack (expected 40, got 0)

In [ ]:
# Processa os arquivos para cada par de coordenadas
for ra, dec in tqdm(zip(ra_list, dec_list), total=len(ra_list), desc="Processing"):

    source_name = create_source_name(ra, dec)
    file_source_name = create_file_source_name(ra, dec)
    plot_lc(output_directory, file_source_name, source_name)

Processing:   0%|          | 0/2106 [00:00<?, ?it/s]

caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0081463341401786_5.3524355182202/ztf_ra0.0081463341401786_dec5.3524355182202_ztf_g.txt
Arquivo ztf_ra0.0081463341401786_dec5.3524355182202_ztf_g.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0081463341401786_5.3524355182202/ztf_ra0.0081463341401786_dec5.3524355182202_ztf_r.txt
Arquivo ztf_ra0.0081463341401786_dec5.3524355182202_ztf_r.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0081463341401786_5.3524355182202/ztf_ra0.0081463341401786_dec5.3524355182202_ztf_i.txt
Arquivo ztf_ra0.0081463341401786_dec5.3524355182202_ztf_i.txt não encontrado. Pulando para a próxima banda.


Processing:   0%|          | 1/2106 [00:00<12:19,  2.85it/s]

caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0739231827065509_2.0077513136953704/ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_g.txt
Arquivo ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_g.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0739231827065509_2.0077513136953704/ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_r.txt
Arquivo ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_r.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0739231827065509_2.0077513136953704/ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_i.txt
Arquivo ztf_ra0.0739231827065509_dec2.0077513136953704_ztf_i.txt não encontrado. Pulando para a próxima banda.


Processing:   0%|          | 2/2106 [00:00<11:53,  2.95it/s]

caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0872570582241678_-3.874658667641907/ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_g.txt
Arquivo ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_g.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0872570582241678_-3.874658667641907/ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_r.txt
Arquivo ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_r.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i0.0872570582241678_-3.874658667641907/ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_i.txt
Arquivo ztf_ra0.0872570582241678_dec-3.874658667641907_ztf_i.txt não encontrado. Pulando para a próxima banda.


... (528 outputs left out)

Processing:  13%|█▎        | 267/2106 [01:35<10:24,  2.94it/s]

caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.163815_-7.987165/ztf_ra28.163815_dec-7.987165_ztf_g.txt
Arquivo ztf_ra28.163815_dec-7.987165_ztf_g.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.163815_-7.987165/ztf_ra28.163815_dec-7.987165_ztf_r.txt
Arquivo ztf_ra28.163815_dec-7.987165_ztf_r.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.163815_-7.987165/ztf_ra28.163815_dec-7.987165_ztf_i.txt
Arquivo ztf_ra28.163815_dec-7.987165_ztf_i.txt não encontrado. Pulando para a próxima banda.


Processing:  13%|█▎        | 268/2106 [01:36<10:24,  2.94it/s]

caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.193628_0.710839/ztf_ra28.193628_dec0.710839_ztf_g.txt
Arquivo ztf_ra28.193628_dec0.710839_ztf_g.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.193628_0.710839/ztf_ra28.193628_dec0.710839_ztf_r.txt
Arquivo ztf_ra28.193628_dec0.710839_ztf_r.txt não encontrado. Pulando para a próxima banda.
caminho para bandas no plot arquivos_de_saida_bandas_g_r_i28.193628_0.710839/ztf_ra28.193628_dec0.710839_ztf_i.txt
Arquivo ztf_ra28.193628_dec0.710839_ztf_i.txt não encontrado. Pulando para a próxima banda.


In [ ]:
# Processa os arquivos para cada par de coordenadas
for ra, dec in tqdm(zip(ra_list, dec_list), total=len(ra_list), desc="Processing"):

    source_name = create_source_name(ra, dec)
    file_source_name = create_file_source_name(ra, dec)   
    plot_hist(output_directory, file_source_name, source_name)

In [ ]:
# Crie o histograma
plt.hist(rms_g, bins=10, color='r', alpha=0.7, label=f'Banda - g ({len(rms_g)})')
plt.hist(rms_r, bins=10, color='g', alpha=0.7, label=f'Banda - r ({len(rms_r)})')
plt.hist(rms_i, bins=10, color='darkorange', alpha=0.7, label=f'Banda - i ({len(rms_i)})')

plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.title('RMS da magnitude')
plt.legend()
plt.grid(True)

# Exiba o histograma
plt.show()


In [ ]:
# Crie o histograma
plt.hist(flux_g, bins=10, color='r', alpha=0.7, label=f'Banda - g ({len(flux_g)})')
plt.hist(flux_r, bins=10, color='g', alpha=0.7, label=f'Banda - r ({len(flux_r)})')
plt.hist(flux_i, bins=10, color='darkorange', alpha=0.7, label=f'Banda - i ({len(flux_i)})')

plt.xlabel('Valores')
plt.ylabel('Frequência')
plt.title('RMS do fluxo')
plt.legend()
plt.grid(True)

# Exiba o histograma
plt.show()
